# Banking Credit Default Risk & Cross-Sell Engine
**Machine Learning Internship Portfolio Project**

---

## Project Overview
This notebook builds an end-to-end ML system that:
1. **Predicts** the probability a loan applicant will default
2. **Recommends** cross-sell banking products to low-risk customers

**Dataset:** Kaggle Credit Risk Dataset (~32,000 rows)  
**Models:** Logistic Regression (baseline) vs XGBoost (production)  
**Explainability:** SHAP values  
**Cross-Sell Engine:** Rule-based segmentation + FP-Growth association rules

---

## 1. Setup & Imports

In [ ]:
# !pip install pandas numpy scikit-learn xgboost shap mlxtend matplotlib seaborn plotly reportlab

import warnings
warnings.filterwarnings('ignore')

import os, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, confusion_matrix
)
from xgboost import XGBClassifier
import shap
from mlxtend.frequent_patterns import fpgrowth, association_rules as mlxtend_rules
from dataclasses import dataclass, field
from typing import List

os.makedirs('artifacts', exist_ok=True)
print('All imports OK')

## 2. Data Loading & Exploration

**Dataset:** Kaggle Credit Risk Dataset  
Download from Kaggle (search: `Credit Risk Dataset laotse`) and place `credit_risk_dataset.csv` in the `data/` folder.

In [ ]:
DATA_PATH = 'data/credit_risk_dataset.csv'

REQUIRED_COLUMNS = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'loan_grade', 'loan_intent', 'home_ownership',
    'cb_person_default_on_file', 'loan_status'
]

df_raw = pd.read_csv(DATA_PATH)
missing_cols = set(REQUIRED_COLUMNS) - set(df_raw.columns)
if missing_cols:
    raise ValueError(f'Missing columns: {missing_cols}')
df = df_raw[REQUIRED_COLUMNS].copy()

print(f'Shape: {df.shape}')
print(f'Default rate: {df["loan_status"].mean():.1%}')
df.head()

In [ ]:
print('=== Data Info ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== Class Distribution ===')
print(df['loan_status'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold')

counts = df['loan_status'].value_counts()
axes[0,0].bar(['No Default (0)', 'Default (1)'], [counts[0], counts[1]], color=['#28a745', '#dc3545'])
axes[0,0].set_title('Class Distribution')
axes[0,0].set_ylabel('Count')

axes[0,1].hist(df['person_age'], bins=30, color='steelblue', edgecolor='white')
axes[0,1].set_title('Age Distribution')

axes[0,2].hist(df['person_income'].clip(upper=200000), bins=30, color='teal', edgecolor='white')
axes[0,2].set_title('Income Distribution (capped 200K)')

grade_default = df.groupby('loan_grade')['loan_status'].mean().reindex(['A','B','C','D','E','F','G'])
axes[1,0].bar(grade_default.index, grade_default.values, color='coral')
axes[1,0].set_title('Default Rate by Loan Grade')

df[df['loan_status']==0]['loan_int_rate'].dropna().hist(ax=axes[1,1], bins=20, alpha=0.6, label='No Default', color='green')
df[df['loan_status']==1]['loan_int_rate'].dropna().hist(ax=axes[1,1], bins=20, alpha=0.6, label='Default', color='red')
axes[1,1].set_title('Interest Rate by Default Status')
axes[1,1].legend()

intent_counts = df['loan_intent'].value_counts()
axes[1,2].barh(intent_counts.index, intent_counts.values, color='purple', alpha=0.7)
axes[1,2].set_title('Loan Intent Distribution')

plt.tight_layout()
plt.savefig('artifacts/eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA chart saved.')

## 3. Feature Engineering

| Transformation | Columns |
|---|---|
| Median imputation | `loan_int_rate`, `person_emp_length` |
| Ordinal encoding | `loan_grade` (A=1 to G=7) |
| One-hot encoding | `loan_intent`, `home_ownership` |
| Binary encoding | `cb_person_default_on_file` |
| Derived feature | `debt_to_income = loan_amnt / person_income` |
| StandardScaler | Numeric columns (Logistic Regression only) |

In [ ]:
GRADE_MAP = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
NUMERIC_COLS = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'debt_to_income'
]
MEDIAN_IMPUTE_COLS = ['loan_int_rate', 'person_emp_length']

class FeatureEngineer:
    def __init__(self):
        self.scaler = StandardScaler()
        self.medians = {}
        self.encoded_columns = []

    def _base_transform(self, X):
        X = X.copy()
        for col in MEDIAN_IMPUTE_COLS:
            X[col] = X[col].fillna(self.medians[col])
        X['debt_to_income'] = (X['loan_amnt'] / X['person_income'].replace(0, np.nan)).fillna(0)
        X['loan_grade'] = X['loan_grade'].map(GRADE_MAP)
        X['cb_person_default_on_file'] = (X['cb_person_default_on_file'] == 'Y').astype(int)
        X = pd.get_dummies(X, columns=['loan_intent', 'home_ownership'])
        return X

    def fit_transform(self, X):
        for col in MEDIAN_IMPUTE_COLS:
            self.medians[col] = X[col].median()
        X_out = self._base_transform(X)
        self.encoded_columns = X_out.columns.tolist()
        return X_out

    def transform(self, X):
        X_out = self._base_transform(X)
        for col in self.encoded_columns:
            if col not in X_out.columns:
                X_out[col] = 0
        return X_out[self.encoded_columns]

    def fit_transform_scaled(self, X):
        X_enc = self.fit_transform(X)
        X_sc = X_enc.copy()
        present = [c for c in NUMERIC_COLS if c in X_sc.columns]
        X_sc[present] = self.scaler.fit_transform(X_enc[present])
        return X_sc

    def transform_scaled(self, X):
        X_enc = self.transform(X)
        X_sc = X_enc.copy()
        present = [c for c in NUMERIC_COLS if c in X_sc.columns]
        X_sc[present] = self.scaler.transform(X_enc[present])
        return X_sc

# Split
X = df.drop(columns=['loan_status'])
y = df['loan_status']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

fe = FeatureEngineer()
X_train_sc  = fe.fit_transform_scaled(X_train)
X_val_sc    = fe.transform_scaled(X_val)
X_test_sc   = fe.transform_scaled(X_test)
X_train_enc = fe.transform(X_train)
X_val_enc   = fe.transform(X_val)
X_test_enc  = fe.transform(X_test)
feature_names = X_test_enc.columns.tolist()
print(f'Features after encoding: {len(feature_names)}')

## 4. Model 1 — Logistic Regression (Baseline)

In [ ]:
class BaselineModel:
    def __init__(self):
        self.model = LogisticRegression(
            solver='lbfgs', max_iter=1000, class_weight='balanced', random_state=42
        )
    def train(self, X, y): self.model.fit(X, y)
    def predict_proba(self, X): return self.model.predict_proba(X)[:, 1]
    def predict(self, X, threshold=0.5): return (self.predict_proba(X) >= threshold).astype(int)

print('Training Logistic Regression...')
lr = BaselineModel()
lr.train(X_train_sc, y_train)
print(f'LR Val AUC: {roc_auc_score(y_val, lr.predict_proba(X_val_sc)):.4f}')

## 5. Model 2 — XGBoost with RandomizedSearchCV & SHAP

In [ ]:
PARAM_DIST = {
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 300, 400],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

class XGBoostModel:
    def __init__(self, scale_pos_weight=1.0):
        self.scale_pos_weight = scale_pos_weight
        self.model = None
        self.explainer = None

    def train(self, X_train, y_train, X_val=None, y_val=None):
        base = XGBClassifier(
            scale_pos_weight=self.scale_pos_weight,
            random_state=42, eval_metric='auc', verbosity=0
        )
        search = RandomizedSearchCV(
            base, PARAM_DIST, n_iter=20, cv=5, scoring='roc_auc',
            random_state=42, n_jobs=-1, verbose=1
        )
        search.fit(X_train, y_train)
        self.model = search.best_estimator_
        self.explainer = shap.TreeExplainer(self.model)
        return search.best_params_

    def predict_proba(self, X): return self.model.predict_proba(X)[:, 1]
    def predict(self, X, threshold=0.5): return (self.predict_proba(X) >= threshold).astype(int)

    def shap_values(self, X):
        vals = self.explainer.shap_values(X)
        return vals[1] if isinstance(vals, list) else vals

scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f'scale_pos_weight = {scale_pos_weight:.2f}')
print('Training XGBoost (20 iter, 5-fold CV) — ~2 minutes...')
xgb = XGBoostModel(scale_pos_weight=scale_pos_weight)
best_params = xgb.train(X_train_enc, y_train, X_val_enc, y_val)
print(f'Best params: {best_params}')
print(f'XGB Val AUC: {roc_auc_score(y_val, xgb.predict_proba(X_val_enc)):.4f}')

## 6. Threshold Selection & Evaluation

In [ ]:
def select_threshold(y_true, y_proba):
    p, r, t = precision_recall_curve(y_true, y_proba)
    f1s = 2 * p * r / (p + r + 1e-9)
    return float(t[int(np.argmax(f1s[:-1]))])

def ks_statistic(y_true, y_proba):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    return float(np.max(tpr - fpr))

def compute_metrics(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'roc_auc':   float(roc_auc_score(y_true, y_proba)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall':    float(recall_score(y_true, y_pred, zero_division=0)),
        'f1':        float(f1_score(y_true, y_pred, zero_division=0)),
        'ks':        ks_statistic(y_true, y_proba),
    }

threshold = select_threshold(y_val, xgb.predict_proba(X_val_enc))
print(f'Selected threshold (max F1 on val): {threshold:.4f}')

lr_metrics  = compute_metrics(y_test, lr.predict_proba(X_test_sc), threshold)
xgb_metrics = compute_metrics(y_test, xgb.predict_proba(X_test_enc), threshold)

results_df = pd.DataFrame({
    'Metric': ['ROC-AUC', 'Precision', 'Recall', 'F1 Score', 'KS Statistic'],
    'Logistic Regression': [f"{lr_metrics[k]:.4f}" for k in ['roc_auc','precision','recall','f1','ks']],
    'XGBoost': [f"{xgb_metrics[k]:.4f}" for k in ['roc_auc','precision','recall','f1','ks']],
})
print('\n=== Test Set Results ===')
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Evaluation — Test Set', fontsize=15, fontweight='bold')

# ROC curves
for name, proba in [('Logistic Regression', lr.predict_proba(X_test_sc)),
                     ('XGBoost', xgb.predict_proba(X_test_enc))]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)
axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves'); axes[0].legend(fontsize=8)

# PR curve
p, r, t = precision_recall_curve(y_test, xgb.predict_proba(X_test_enc))
axes[1].plot(r[:-1], p[:-1], linewidth=2, color='darkorange')
idx = int(np.argmin(np.abs(t - threshold)))
axes[1].scatter([r[idx]], [p[idx]], color='red', s=100, zorder=5, label=f'Threshold={threshold:.2f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curve (XGBoost)'); axes[1].legend()

# Confusion matrix
xgb_pred = xgb.predict(X_test_enc, threshold)
cm = confusion_matrix(y_test, xgb_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['No Default','Default'], yticklabels=['No Default','Default'])
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
axes[2].set_title('Confusion Matrix (XGBoost)')

plt.tight_layout()
plt.savefig('artifacts/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: artifacts/model_evaluation.png')

In [ ]:
print('Computing SHAP values...')
shap_vals = xgb.shap_values(X_test_enc)
mean_abs = np.abs(shap_vals).mean(axis=0)
top_idx = np.argsort(mean_abs)[-10:]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh([feature_names[i] for i in top_idx], mean_abs[top_idx], color='steelblue')
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Top 10 Feature Importances (SHAP)', fontweight='bold')
plt.tight_layout()
plt.savefig('artifacts/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: artifacts/shap_importance.png')

## 7. Cross-Sell Engine — Step 1: Rule-Based Segmentation

Only customers with predicted default probability **< 30%** are eligible.

| Segment | Criteria | Products |
|---|---|---|
| Young Starter | Age < 30, Income < 40K | Student Loan, Secured Credit Card, Savings Account |
| Rising Professional | Age 30-45, Income 40K-80K | Personal Loan, Travel Credit Card, Mutual Funds |
| Established Earner | Age 30-55, Income > 80K | Home Loan, Premium Credit Card, Investment Portfolio |
| Senior Stable | Age > 55 | Fixed Deposits, Insurance, Retirement Fund |

In [ ]:
@dataclass
class Segment:
    name: str
    description: str
    products: List[str] = field(default_factory=list)

SEGMENTS = {
    'young_starter':       Segment('Young Starter', 'Age<30, Income<40K — early career',
                                   ['Student Loan','Secured Credit Card','Savings Account']),
    'rising_professional': Segment('Rising Professional', 'Age 30-45, Income 40K-80K — growth stage',
                                   ['Personal Loan','Travel Credit Card','Mutual Funds']),
    'established_earner':  Segment('Established Earner', 'Age 30-55, Income>80K — wealth building',
                                   ['Home Loan','Premium Credit Card','Investment Portfolio']),
    'senior_stable':       Segment('Senior Stable', 'Age>55 — preservation & retirement',
                                   ['Fixed Deposits','Insurance','Retirement Fund']),
}

def assign_segment(age, income):
    if age > 55: return SEGMENTS['senior_stable']
    if age < 30 and income < 40_000: return SEGMENTS['young_starter']
    if 30 <= age <= 45 and 40_000 <= income <= 80_000: return SEGMENTS['rising_professional']
    if 30 <= age <= 55 and income > 80_000: return SEGMENTS['established_earner']
    if income < 40_000: return SEGMENTS['young_starter']
    if income <= 80_000: return SEGMENTS['rising_professional']
    return SEGMENTS['established_earner']

for age, income in [(25,30000),(35,60000),(45,100000),(60,50000)]:
    s = assign_segment(age, income)
    print(f'Age={age}, Income={income:,} => {s.name}: {s.products}')

## 8. Cross-Sell Engine — Step 2: FP-Growth Association Rules

A **synthetic transaction table** maps each customer's `loan_intent` and `loan_grade`
to banking products, simulating realistic co-occurrence patterns.

In [ ]:
INTENT_PRODUCT_MAP = {
    'EDUCATION':         ['student_loan', 'credit_card'],
    'MEDICAL':           ['health_insurance', 'personal_loan'],
    'VENTURE':           ['business_loan', 'investment_account'],
    'PERSONAL':          ['personal_loan', 'credit_card'],
    'HOMEIMPROVEMENT':   ['home_loan', 'savings_account'],
    'DEBTCONSOLIDATION': ['balance_transfer_card', 'personal_loan'],
}
GRADE_PRODUCT_MAP = {'A': ['investment_account','premium_credit_card'], 'B': ['investment_account'], 'C': ['savings_account']}
_ALL_PRODUCTS = sorted({p for v in INTENT_PRODUCT_MAP.values() for p in v} | {p for v in GRADE_PRODUCT_MAP.values() for p in v})

def build_transaction_df(df_input):
    rows = []
    for _, row in df_input.iterrows():
        basket = set()
        basket.update(INTENT_PRODUCT_MAP.get(str(row.get('loan_intent','')).upper(), []))
        basket.update(GRADE_PRODUCT_MAP.get(str(row.get('loan_grade','')), []))
        rows.append({p: (p in basket) for p in _ALL_PRODUCTS})
    return pd.DataFrame(rows, dtype=bool)

def mine_rules(df_input):
    transactions = build_transaction_df(df_input)
    frequent = fpgrowth(transactions, min_support=0.05, use_colnames=True)
    if frequent.empty: return pd.DataFrame()
    rules = mlxtend_rules(frequent, metric='confidence', min_threshold=0.4)
    return rules.sort_values('lift', ascending=False).reset_index(drop=True)

def filter_rules_for_products(rules, products):
    if rules.empty: return rules
    product_set = {p.lower().replace(' ','_') for p in products}
    mask = rules['antecedents'].apply(lambda x: bool(set(x) & product_set))
    return rules[mask].head(5).reset_index(drop=True)

print('Mining association rules...')
rules = mine_rules(df)
print(f'Rules mined: {len(rules)}')
if not rules.empty:
    disp = rules.copy()
    disp['antecedents'] = disp['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    disp['consequents'] = disp['consequents'].apply(lambda x: ', '.join(sorted(x)))
    print(disp[['antecedents','consequents','support','confidence','lift']].head(10).round(3).to_string(index=False))

## 9. End-to-End Demo — Single Customer Prediction

In [ ]:
def predict_and_recommend(age, income, emp_length, loan_amnt, loan_int_rate,
                           loan_grade, loan_intent, home_ownership, cb_default='N'):
    loan_pct_inc = round(loan_amnt / max(income, 1), 4)
    customer = pd.DataFrame([{
        'person_age': age, 'person_income': income, 'person_emp_length': emp_length,
        'loan_amnt': loan_amnt, 'loan_int_rate': loan_int_rate,
        'loan_percent_income': loan_pct_inc, 'loan_grade': loan_grade,
        'loan_intent': loan_intent, 'home_ownership': home_ownership,
        'cb_person_default_on_file': cb_default,
    }])
    X_enc = fe.transform(customer)
    X_sc  = fe.transform_scaled(customer)
    risk  = float(xgb.predict_proba(X_enc)[0])
    lr_r  = float(lr.predict_proba(X_sc)[0])

    print('=' * 55)
    print('         CREDIT RISK & CROSS-SELL REPORT')
    print('=' * 55)
    print(f'  Age:{age}  Income:${income:,}  Loan:${loan_amnt:,}  Grade:{loan_grade}')
    print('-' * 55)
    print(f'  XGBoost Risk Score : {risk:.1%}')
    print(f'  LR Risk Score      : {lr_r:.1%}')
    label = 'LOW' if risk < 0.3 else ('MEDIUM' if risk < 0.5 else 'HIGH')
    print(f'  Risk Level         : {label}')
    print('-' * 55)
    sv = xgb.shap_values(X_enc)[0]
    fname = X_enc.columns.tolist()
    print('  Top 5 SHAP Factors:')
    for i in np.argsort(np.abs(sv))[-5:][::-1]:
        print(f'    {"+" if sv[i]>0 else "-"} {fname[i]}: {sv[i]:+.3f}')
    if risk < 0.3:
        print('-' * 55)
        seg = assign_segment(age, income)
        print(f'  Segment : {seg.name}')
        print(f'  Products: {" | ".join(seg.products)}')
        boosted = filter_rules_for_products(rules, seg.products)
        if not boosted.empty:
            print('  Association Rule Boosts:')
            for _, r in boosted.head(3).iterrows():
                a = ', '.join(sorted(r['antecedents'])); c = ', '.join(sorted(r['consequents']))
                print(f'    {a} => {c} (conf={r["confidence"]:.2f}, lift={r["lift"]:.2f})')
    else:
        print(f'  Cross-sell: NOT eligible (risk >= 30%)')
    print('=' * 55)

print('CUSTOMER 1 — Low Risk Young Starter')
predict_and_recommend(25, 35000, 2, 5000, 9.5, 'B', 'EDUCATION', 'RENT', 'N')

print('\nCUSTOMER 2 — High Risk')
predict_and_recommend(28, 20000, 1, 15000, 22.0, 'F', 'DEBTCONSOLIDATION', 'RENT', 'Y')

print('\nCUSTOMER 3 — Low Risk Established Earner')
predict_and_recommend(42, 120000, 15, 30000, 7.0, 'A', 'HOMEIMPROVEMENT', 'MORTGAGE', 'N')

## 10. Save Artifacts

In [ ]:
for name, obj in [('fe', fe), ('lr', lr), ('xgb', xgb), ('rules', rules)]:
    with open(f'artifacts/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)

with open('artifacts/metadata.pkl', 'wb') as f:
    pickle.dump({'threshold': threshold, 'lr_metrics': lr_metrics,
                 'xgb_metrics': xgb_metrics, 'feature_names': feature_names}, f)

print('Artifacts saved:')
for fname in sorted(os.listdir('artifacts')):
    print(f'  artifacts/{fname}')

## 11. Results Summary

In [ ]:
print('=== FINAL TEST SET RESULTS ===')
print(results_df.to_string(index=False))
print(f'\nSelected threshold: {threshold:.4f}')
print(f'AUC improvement over LR baseline: +{xgb_metrics["roc_auc"] - lr_metrics["roc_auc"]:.4f}')

## 12. Business Impact & Conclusion

### Business Impact
Assuming a mid-sized bank processing **10,000 loan applications/month** with **average loan $15,000**:

- **Default Detection:** The model flags the majority of likely defaulters before approval, reducing bad-loan exposure significantly.
- **Cross-Sell Revenue:** ~78% of applicants are low-risk and eligible. A 5% conversion on 7,800 eligible customers/month represents meaningful incremental revenue from fees and interest margins.

### Conclusion
This project demonstrates a production-ready dual-objective ML pipeline:
1. **Credit risk scoring** with explainable per-prediction SHAP analysis
2. **Cross-sell recommendation engine** combining rule-based segmentation with FP-Growth association rules

### Next Steps
- Replace synthetic product basket with real customer transaction history
- Add data drift monitoring for production deployment
- A/B test recommendations vs. existing manual process
- Deploy via Streamlit Cloud or Docker with authentication

---
*Banking Credit Default Risk & Cross-Sell Engine | Internship Portfolio Project*